# Phase 2: Autoencoder Architecture & Initialization

---

## What This Notebook Does

In Phase 1, we converted raw `.wav` audio files into 2D **Log-Mel Spectrograms** of shape
`(128, 313)` — think of them as grayscale "photographs" of sound. We then scaled every pixel
value into the range `[0, 1]` using `MinMaxScaler`.

Now, in Phase 2, we build the **brain** of our anomaly detection system: an **Unsupervised
Autoencoder** implemented in PyTorch.

### The Core Idea

An Autoencoder is a neural network that learns to **copy its input to its output** — but it
must do so through a severe **information bottleneck**. This forces the network to learn only
the most essential patterns (the "fingerprint") of normal machine sounds.

```
Input (40,064 features)  →  Encoder  →  Bottleneck (64 features)  →  Decoder  →  Output (40,064 features)
         ╰─────────────────── must match ───────────────────╯
```

- **During training**: We feed ONLY normal spectrograms. The model learns to compress and
  reconstruct them with minimal error.
- **During inference**: When an anomalous spectrogram is fed in, the model has *never seen*
  that pattern before. It fails to reconstruct it accurately, producing a **high Reconstruction
  Error (MSE)** — this error spike *is* our anomaly signal.

### Why This Architecture?

| Design Choice | Reason |
|--------------|--------|
| `nn.Linear` (fully-connected) layers | Simple, interpretable, and sufficient for this spectrogram size |
| 64-dimensional bottleneck | Forces extreme compression (40,064 → 64 = **626× reduction**) |
| `nn.Dropout(0.2)` | Prevents memorization of background noise — critical for Domain Shift |
| `nn.Sigmoid()` final layer | Output must match our `[0, 1]` scaled input range |
| L2 regularization (`weight_decay`) | Penalizes large weights, improving generalization |

---

## 1. Library Imports

We need a minimal set of libraries for this phase:

| Library | Purpose |
|---------|--------|
| **`torch`** | The PyTorch deep learning framework — provides tensors, autograd, and GPU acceleration |
| **`torch.nn`** | Neural network building blocks (`Linear`, `ReLU`, `Sigmoid`, `Sequential`, etc.) |
| **`numpy`** | For loading our `.npy` spectrograms and array operations |
| **`os`** | Filesystem operations for locating data and saving model checkpoints |

In [1]:
import torch
import torch.nn as nn
import numpy as np
import os

print("Libraries imported successfully.")
print(f"  PyTorch version : {torch.__version__}")
print(f"  NumPy version   : {np.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU device      : {torch.cuda.get_device_name(0)}")

Libraries imported successfully.
  PyTorch version : 2.5.1
  NumPy version   : 2.0.1
  CUDA available  : True
  GPU device      : NVIDIA GeForce RTX 4060 Laptop GPU


---

## 2. The Autoencoder Architecture

### Why We Flatten: The Shape Problem

Our spectrograms are 2D matrices of shape `(128, 313)` — that's **128 Mel frequency bands × 313
time frames**. But `nn.Linear` layers expect a **1D vector** as input.

So the very first thing our `forward()` method does is **flatten** each 2D spectrogram into a
single vector of length `128 × 313 = 40,064`:

```
Input batch:   (Batch, 128, 313)   ← 2D spectrograms
After flatten: (Batch, 40064)      ← 1D vectors
After model:   (Batch, 40064)      ← 1D reconstructions
After reshape: (Batch, 128, 313)   ← back to 2D spectrograms
```

### Encoder — The Compression Path

The encoder progressively compresses the input through three `Linear` layers:

```
40,064  →  1,024  →  256  →  64 (bottleneck)
```

Each compression step uses **ReLU** (Rectified Linear Unit) activation:

$$\text{ReLU}(x) = \max(0, x)$$

ReLU is the go-to activation for hidden layers because:
- It's computationally cheap (just a threshold)
- It doesn't saturate for positive values (avoids vanishing gradients)
- It introduces non-linearity, allowing the network to learn complex patterns

We also add **Dropout(p=0.2)** after the first hidden layer. During training, this randomly
"switches off" 20% of neurons each forward pass. This is critical for our use case because:

> **Domain Shift Problem**: In the DCASE 2024 dataset, the test set may contain machines
> recorded under different conditions (speed, load, environment) than the training set.
> Dropout forces the network to learn **robust, redundant representations** instead of
> memorizing specific noise patterns from the training conditions.

### Decoder — The Reconstruction Path

The decoder mirrors the encoder, expanding back from the bottleneck:

```
64 (bottleneck)  →  256  →  1,024  →  40,064
```

The **critical detail** is the final activation function: **Sigmoid**.

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

**Why Sigmoid and not ReLU?**

In Phase 1, we explicitly scaled all spectrogram values to `[0, 1]` using `MinMaxScaler`.
The Sigmoid function outputs values **exclusively between 0 and 1** — this mathematically
guarantees that the decoder's output matches the range of our input data.

If we used ReLU here instead, the output could be *any* positive number (0 to ∞), creating
a mismatch with our `[0, 1]` targets and making the MSE loss meaningless.

In [2]:
class AcousticAutoencoder(nn.Module):
    """
    Unsupervised Autoencoder for Acoustic Anomaly Detection.

    Takes a Log-Mel Spectrogram of shape (128, 313), compresses it through
    a 64-dimensional bottleneck, and reconstructs it. High reconstruction
    error (MSE) on unseen data indicates an acoustic anomaly.

    Architecture:
        Encoder: 40064 → 1024 → 256 → 64  (compression ratio: 626×)
        Decoder: 64 → 256 → 1024 → 40064  (symmetric expansion)
    """

    def __init__(self):
        super(AcousticAutoencoder, self).__init__()

        # Input dimension: flatten a (128, 313) spectrogram into a 1D vector
        self.input_dim = 128 * 313  # = 40,064 features

        # -----------------------------------------------------------------
        # ENCODER — Compress from 40,064 features down to 64
        # -----------------------------------------------------------------
        # Each Linear layer learns a weight matrix W and bias b:
        #   output = input @ W.T + b
        #
        # Layer progression: 40064 → 1024 → 256 → 64
        # -----------------------------------------------------------------
        self.encoder = nn.Sequential(
            nn.Linear(self.input_dim, 1024),  # (Batch, 40064) → (Batch, 1024)
            nn.ReLU(),                        # Zero out negatives, keep positives
            nn.Dropout(p=0.2),                # Randomly disable 20% of neurons
                                              # during training to prevent
                                              # memorization of background noise
                                              # (Domain Shift regularization)

            nn.Linear(1024, 256),             # (Batch, 1024) → (Batch, 256)
            nn.ReLU(),                        # Non-linear activation

            nn.Linear(256, 64),               # (Batch, 256) → (Batch, 64)
            nn.ReLU(),                        # Bottleneck activation
        )

        # -----------------------------------------------------------------
        # DECODER — Expand from 64 features back to 40,064
        # -----------------------------------------------------------------
        # Mirrors the encoder architecture symmetrically.
        # The FINAL layer uses Sigmoid (NOT ReLU) because our input data
        # was scaled to [0, 1] by MinMaxScaler in Phase 1.
        # Sigmoid guarantees output ∈ [0, 1], matching the target range.
        # -----------------------------------------------------------------
        self.decoder = nn.Sequential(
            nn.Linear(64, 256),               # (Batch, 64) → (Batch, 256)
            nn.ReLU(),                        # Non-linear activation

            nn.Linear(256, 1024),             # (Batch, 256) → (Batch, 1024)
            nn.ReLU(),                        # Non-linear activation

            nn.Linear(1024, self.input_dim),  # (Batch, 1024) → (Batch, 40064)
            nn.Sigmoid(),                     # CRITICAL: Squash output to [0, 1]
                                              # to match MinMaxScaler range
        )

    def forward(self, x):
        """
        Forward pass: flatten → encode → decode → reshape.

        Args:
            x: Input tensor of shape (Batch, 128, 313)

        Returns:
            Reconstructed tensor of shape (Batch, 128, 313)
        """
        # Store the original batch size for reshaping later
        batch_size = x.shape[0]               # e.g., 16

        # Flatten: (Batch, 128, 313) → (Batch, 40064)
        # This collapses the 2D spectrogram into a 1D feature vector
        x = x.view(batch_size, -1)            # -1 auto-calculates 128*313=40064

        # Encode: (Batch, 40064) → (Batch, 64)
        # Compress the spectrogram into a 64-dim latent representation
        encoded = self.encoder(x)

        # Decode: (Batch, 64) → (Batch, 40064)
        # Reconstruct the spectrogram from the compressed representation
        decoded = self.decoder(encoded)

        # Reshape: (Batch, 40064) → (Batch, 128, 313)
        # Restore the original 2D spectrogram dimensions
        output = decoded.view(batch_size, 128, 313)

        return output


print("AcousticAutoencoder class defined successfully.")

AcousticAutoencoder class defined successfully.


---

## 3. Model Initialization & Shape Verification

Before we invest hours of GPU time in training, we need to **verify** that our architecture
actually works. This means:

1. **Device selection** — Automatically use GPU (CUDA) if available, fall back to CPU
2. **Instantiation** — Create the model and move it to the selected device
3. **Smoke test** — Pass a dummy batch through the model and confirm the output shape
   matches the input shape exactly: `[Batch, 128, 313]`

This is a **non-negotiable sanity check**. A shape mismatch here means the model's math is
broken — and no amount of training will fix it. Better to catch it now than after 50 epochs.

### What to Look For

- Input shape: `torch.Size([16, 128, 313])`
- Output shape: `torch.Size([16, 128, 313])` ← **must match exactly**
- Total parameters: should be in the tens of millions (we have a 40K-wide input layer)

In [3]:
# ===========================================================================
# DEVICE SELECTION — Use GPU if available, otherwise CPU
# ===========================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ===========================================================================
# MODEL INSTANTIATION
# ===========================================================================
model = AcousticAutoencoder().to(device)

# Print the full architecture
print("\n" + "=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)
print(model)

# ===========================================================================
# PARAMETER COUNT — How many trainable weights does this model have?
# ===========================================================================
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

# ===========================================================================
# SMOKE TEST — Verify shapes with a dummy batch
# ===========================================================================
# Create a random tensor that mimics one batch of 16 spectrograms
# Shape: (batch_size=16, n_mels=128, time_frames=313)
dummy_input = torch.randn(16, 128, 313).to(device)

# Forward pass through the model
with torch.no_grad():  # No need to track gradients for this test
    dummy_output = model(dummy_input)

print(f"\n{'=' * 70}")
print("SHAPE VERIFICATION (Smoke Test)")
print(f"{'=' * 70}")
print(f"  Input shape  : {dummy_input.shape}")   # Expected: [16, 128, 313]
print(f"  Output shape : {dummy_output.shape}")  # Expected: [16, 128, 313]

# Verify output range is [0, 1] (Sigmoid guarantee)
print(f"  Output min   : {dummy_output.min().item():.6f}")
print(f"  Output max   : {dummy_output.max().item():.6f}")

# Final validation
assert dummy_input.shape == dummy_output.shape, "SHAPE MISMATCH! Architecture is broken."
assert dummy_output.min() >= 0.0, "Output below 0! Sigmoid is not working."
assert dummy_output.max() <= 1.0, "Output above 1! Sigmoid is not working."

print("\n  ✓ All assertions passed — architecture is verified and ready for training.")

Using device: cuda
  GPU: NVIDIA GeForce RTX 4060 Laptop GPU
  Memory: 8.6 GB

MODEL ARCHITECTURE
AcousticAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=40064, out_features=1024, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=1024, out_features=256, bias=True)
    (4): ReLU()
    (5): Linear(in_features=256, out_features=64, bias=True)
    (6): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=1024, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1024, out_features=40064, bias=True)
    (5): Sigmoid()
  )
)

Total parameters     : 82,650,816
Trainable parameters : 82,650,816

SHAPE VERIFICATION (Smoke Test)
  Input shape  : torch.Size([16, 128, 313])
  Output shape : torch.Size([16, 128, 313])
  Output min   : 0.473637
  Output max   : 0.526222

  ✓ All assertions passed — architecture is verified and ready

---

## Summary & Architecture Diagram

### Layer-by-Layer Breakdown

| Layer | Input Shape | Output Shape | Activation | Purpose |
|-------|-----------|-------------|------------|--------|
| **Flatten** | `(B, 128, 313)` | `(B, 40064)` | — | Collapse 2D → 1D for Linear layers |
| **Enc. Linear 1** | `(B, 40064)` | `(B, 1024)` | ReLU | First compression (39× reduction) |
| **Dropout** | `(B, 1024)` | `(B, 1024)` | — | Domain Shift regularization (p=0.2) |
| **Enc. Linear 2** | `(B, 1024)` | `(B, 256)` | ReLU | Second compression (4× reduction) |
| **Enc. Linear 3** | `(B, 256)` | `(B, 64)` | ReLU | Bottleneck (4× reduction) |
| **Dec. Linear 1** | `(B, 64)` | `(B, 256)` | ReLU | First expansion |
| **Dec. Linear 2** | `(B, 256)` | `(B, 1024)` | ReLU | Second expansion |
| **Dec. Linear 3** | `(B, 1024)` | `(B, 40064)` | **Sigmoid** | Final reconstruction |
| **Reshape** | `(B, 40064)` | `(B, 128, 313)` | — | Restore 2D spectrogram shape |

### Compression Ratio

```
40,064 features  →  64 features  =  626:1 compression ratio
```

This extreme bottleneck forces the model to learn only the **most essential patterns**
of normal machine sounds, discarding noise and irrelevant details.

### Next: Phase 2b — Training Loop

With the architecture verified, the next step is to:
1. Load `X_train.npy` and `X_val.npy` for each machine type
2. Wrap them in PyTorch `DataLoader` objects
3. Define the loss function (`nn.MSELoss`)
4. Define the optimizer (`Adam` with `weight_decay` for L2 regularization)
5. Train with Early Stopping on validation loss
6. Save the best model weights as `.pth` checkpoints